# ACL Graph Editor
Inspect, delete nodes, mark flag/spawn nodes, and save the ACL PRM graph.

**Workflow:**
1. Run **Cell 1** — loads graph + renders interactive Plotly map (hover to see node IDs)
2. Run **Cell 2** — delete nodes by ID
3. Run **Cell 3** — mark flag hypothesis nodes
4. Run **Cell 4** — mark Blue / Red spawn nodes
5. Run **Cell 5** — save updated graph

In [1]:
# ── Cell 1: Load & Display ──────────────────────────────────────────────────
import pickle, networkx as nx
import numpy as np
import plotly.graph_objects as go
from PIL import Image

GRAPH_PATH = 'acl_graph.pkl'
PGM_PATH   = 'acl_merged.pgm'

with open(GRAPH_PATH, 'rb') as f:
    G = pickle.load(f)

img = np.array(Image.open(PGM_PATH).convert('L'))
H, W = img.shape
res  = 0.15
ox, oy = -20.2, -35.6

def px_to_world(col, row):
    return ox + col * res, oy + (H - 1 - row) * res

# ── Build Plotly figure ──────────────────────────────────────────────────────
fig = go.Figure()

# Background map as a heatmap
fig.add_trace(go.Heatmap(
    z=img[::-1],          # flip so y increases upward
    colorscale='Gray',
    showscale=False,
    x0=ox, dx=res,
    y0=oy, dy=res,
    hoverinfo='skip',
))

# Edges
edge_x, edge_y = [], []
for u, v in G.edges():
    edge_x += [G.nodes[u]['x'], G.nodes[v]['x'], None]
    edge_y += [G.nodes[u]['y'], G.nodes[v]['y'], None]
fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(color='steelblue', width=0.8),
    hoverinfo='skip',
    name='edges',
))

# Nodes
node_x = [G.nodes[n]['x'] for n in G.nodes()]
node_y = [G.nodes[n]['y'] for n in G.nodes()]
node_ids = list(G.nodes())
node_text = [
    f'node {n}<br>x={G.nodes[n]["x"]:.2f}, y={G.nodes[n]["y"]:.2f}<br>degree={G.degree(n)}'
    for n in G.nodes()
]
fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    marker=dict(color='red', size=6, line=dict(color='white', width=0.5)),
    text=[str(n) for n in G.nodes()],
    textposition='top right',
    textfont=dict(size=7, color='yellow'),
    hovertext=node_text,
    hoverinfo='text',
    name='nodes',
))

fig.update_layout(
    title=f'ACL PRM — {G.number_of_nodes()} nodes, {G.number_of_edges()} edges',
    width=900, height=900,
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='#888',
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(x=0.01, y=0.99),
)

fig.show()
print(f'Loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print('Hover over nodes to see IDs. Zoom/pan freely.')

Loaded: 297 nodes, 500 edges
Hover over nodes to see IDs. Zoom/pan freely.


In [2]:
# ── Cell 2: Delete nodes ─────────────────────────────────────────────────────
# Add node IDs to this list and run the cell.

NODES_TO_DELETE = [255, 4, 89, 221, 205, 115, 141]   # e.g. [12, 47, 203]

if NODES_TO_DELETE:
    missing = [n for n in NODES_TO_DELETE if n not in G]
    if missing:
        print(f'WARNING — not in graph: {missing}')
    to_remove = [n for n in NODES_TO_DELETE if n in G]
    for n in to_remove:
        d = G.nodes[n]
        print(f'  removing node {n}: x={d["x"]:.2f}, y={d["y"]:.2f}, degree={G.degree(n)}')
    G.remove_nodes_from(to_remove)
    for i, n in enumerate(G.nodes()):
        G.nodes[n]['node_idx'] = i
    comps = list(nx.connected_components(G))
    print(f'\nGraph now: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, '
          f'{len(comps)} component(s), LCC={len(max(comps, key=len))}')
    print('Re-run Cell 1 to refresh the map.')
else:
    print('NODES_TO_DELETE is empty — nothing removed.')

WARNING — not in graph: [255, 4, 89, 221, 205, 115, 141]

Graph now: 297 nodes, 500 edges, 1 component(s), LCC=297
Re-run Cell 1 to refresh the map.


In [10]:
# ── Cell 3: Flag hypothesis nodes ────────────────────────────────────────────
# Set the node IDs you want to designate as flag hypothesis locations.
# These are stored as a graph-level attribute and highlighted on re-render.

FLAG_HYPOTHESIS_NODES = [48, 151, 20]   # e.g. [5, 88, 201]

if FLAG_HYPOTHESIS_NODES:
    missing = [n for n in FLAG_HYPOTHESIS_NODES if n not in G]
    if missing:
        print(f'WARNING — not in graph: {missing}')
    valid = [n for n in FLAG_HYPOTHESIS_NODES if n in G]
    G.graph['flag_hypothesis_nodes'] = valid
    print('Flag hypothesis nodes set:')
    for n in valid:
        d = G.nodes[n]
        print(f'  node {n}: x={d["x"]:.2f}, y={d["y"]:.2f}')

    # Highlight on map
    fig2 = go.Figure(fig)  # copy last figure
    fh_x = [G.nodes[n]['x'] for n in valid]
    fh_y = [G.nodes[n]['y'] for n in valid]
    fig2.add_trace(go.Scatter(
        x=fh_x, y=fh_y,
        mode='markers+text',
        marker=dict(symbol='star', color='lime', size=16,
                    line=dict(color='black', width=1)),
        text=[str(n) for n in valid],
        textposition='top right',
        textfont=dict(size=9, color='lime'),
        name='flag hypotheses',
        hovertext=[f'FLAG node {n}<br>x={G.nodes[n]["x"]:.2f}, y={G.nodes[n]["y"]:.2f}' for n in valid],
        hoverinfo='text',
    ))
    fig2.update_layout(title=f'ACL PRM — flag hypotheses: {valid}')
    fig2.show()
else:
    print('FLAG_HYPOTHESIS_NODES is empty.')
    existing = G.graph.get('flag_hypothesis_nodes', [])
    if existing:
        print(f'Currently stored in graph: {existing}')

Flag hypothesis nodes set:
  node 48: x=-5.65, y=-18.50
  node 151: x=3.35, y=-18.50
  node 20: x=14.90, y=-14.90


In [6]:
# ── Cell 4: Spawn nodes ──────────────────────────────────────────────────────
# Set node IDs for each team's 3 spawn positions.
# These are saved as graph-level attributes and highlighted on the map.

BLUE_SPAWN_NODES = [268, 270, 146]   # left-of-centre cluster (current defaults)
RED_SPAWN_NODES  = [294, 265, 207]  # right-of-centre cluster (current defaults)

all_spawn = {'blue': BLUE_SPAWN_NODES, 'red': RED_SPAWN_NODES}
for team, nodes in all_spawn.items():
    missing = [n for n in nodes if n not in G]
    if missing:
        print(f'WARNING — {team} nodes not in graph: {missing}')
    valid = [n for n in nodes if n in G]
    G.graph[f'{team}_spawn_nodes'] = valid
    print(f'{team.upper()} spawn nodes:')
    for n in valid:
        d = G.nodes[n]
        print(f'  node {n}: x={d["x"]:.2f}, y={d["y"]:.2f}')

# Highlight on map
fig3 = go.Figure(fig)
for team, color, nodes in [('BLUE', 'dodgerblue', BLUE_SPAWN_NODES),
                             ('RED',  'tomato',     RED_SPAWN_NODES)]:
    valid = [n for n in nodes if n in G]
    fig3.add_trace(go.Scatter(
        x=[G.nodes[n]['x'] for n in valid],
        y=[G.nodes[n]['y'] for n in valid],
        mode='markers+text',
        marker=dict(symbol='square', color=color, size=14,
                    line=dict(color='white', width=1)),
        text=[str(n) for n in valid],
        textposition='top right',
        textfont=dict(size=9, color=color),
        name=f'{team} spawns',
        hovertext=[f'{team} spawn node {n}<br>x={G.nodes[n]["x"]:.2f}, y={G.nodes[n]["y"]:.2f}'
                   for n in valid],
        hoverinfo='text',
    ))
fig3.update_layout(title=f'ACL PRM — spawn nodes (blue={BLUE_SPAWN_NODES}, red={RED_SPAWN_NODES})')
fig3.show()


BLUE spawn nodes:
  node 268: x=-5.80, y=-10.55
  node 270: x=3.95, y=-10.85
  node 146: x=6.80, y=-11.60
RED spawn nodes:
  node 294: x=12.20, y=-11.15
  node 265: x=15.65, y=-12.20
  node 207: x=11.15, y=-12.35


In [5]:
# ── Cell 5: Save ─────────────────────────────────────────────────────────────
with open(GRAPH_PATH, 'wb') as f:
    pickle.dump(G, f)

fh = G.graph.get('flag_hypothesis_nodes', [])
bs = G.graph.get('blue_spawn_nodes', [])
rs = G.graph.get('red_spawn_nodes', [])
print(f'Saved {GRAPH_PATH}')
print(f'  nodes: {G.number_of_nodes()}  edges: {G.number_of_edges()}')
print(f'  flag_hypothesis_nodes: {fh}')
print(f'  blue_spawn_nodes:      {bs}')
print(f'  red_spawn_nodes:       {rs}')


Saved acl_graph.pkl
  nodes: 297  edges: 500
  flag_hypothesis_nodes: [48, 151, 20]
  blue_spawn_nodes:      [268, 270, 146]
  red_spawn_nodes:       [294, 284, 207]
